# Ploting Datata
To imporove the reading of the code the functions used for plotting are in the file [Plotting_functions.py](./Plotting_functions.py)

In [74]:
from locale import normalize
from venv import create

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import warnings

# from Plotting_functions import *  # THIS IS NOT WORKING PROPERLY

In [75]:
# Plotting functions
def plot_protein_phosphosites_together(df, proteins=list, legend=list, path=str, saving_path=str,
                                       saving_info="", save_pdf=False, save_png=False, plot_close=False,
                                       fit_y_lims=False):
    '''Plot to PDF ALL phosphosites of a protein together in one single plot. "y" limits are ajusted to the limits of each phosphosites'''
    # Check the saving folder exists
    saving_folder = re.sub("Data.*", saving_path, path)
    if not os.path.exists(saving_folder):
        print("Creating saving folder")
        os.makedirs(saving_folder)

    for protein in proteins:

        # Create sub-dataframe with only the protein we are interested in. If the protein doesn't exist in the dataframe skip code
        if protein in df['prot_name'].to_list():
            sub_df = df.loc[df['prot_name'] == protein].copy()
            print(f"Ploting sites of protein {protein}")
        elif protein in df['protein_ID'].to_list():
            sub_df = df.loc[df['protein_ID'] == protein].copy()
            print(f"Ploting sites of protein {protein}")
        else:
            print(f"The protein {protein} is not present in the dataset")
            continue
        # Check if a folder for the desired protein exists. If no, create one
        to_save = re.sub("Data.*", saving_path, path)
        if protein in os.listdir(to_save):
            pass
        else:
            new_path = f"{to_save}/{protein}"
            print(f"Createating saving folder for {protein}")
            os.makedirs(new_path)

        sub_df.sort_values(by=['site'], inplace=True)

        # Geting some basic information and parameters for the plots
        number_phos = len(sub_df)
        sqrt_n_p = int(np.ceil(np.sqrt(number_phos)))
        if sqrt_n_p <= 2:
            empty_plots = 0
        else:
            empty_plots = (sqrt_n_p * sqrt_n_p) - number_phos
        # empty_plots = (sqrt_n_p * sqrt_n_p) - number_phos

        # If the protein only has one phosphosite the code cannot make a plot with one subplot, so I'll have to make it individually
        # single_phsite_proteins = []
        # if number_phos == 1:
        #     single_phsite_proteins.append(protein)
        #     print(f"Protein {protein} has only one phosphosite. Plot it individually")
        #     continue

        # Avoid getting rows with empty plots
        if empty_plots >= sqrt_n_p:
            sqrt_n_p_X = sqrt_n_p - 1
        else:
            sqrt_n_p_X = sqrt_n_p

        # Decide wether to fit the y axes or not
        if fit_y_lims == True:  # Fit "y" limits for each phosphosite
            y_fixed = "_y_axis_fixed"
        elif type(fit_y_lims) == list:
            y_lim_min = fit_y_lims[0]
            y_lim_max = fit_y_lims[1]
            y_fixed = f"_y_axis_fixed_{y_lim_min}_{y_lim_max}"
        else:  # Fit the same "y" limit for all phosphosites
            sub_values_df = sub_df.iloc[:, 3:24]
            y_lim_max = (sub_values_df.max().max()) * 1.1
            y_lim_min = (sub_values_df.min().min()) * 1.1
            y_fixed = "_y_axis_general"

        k = 0  # Counter to stop ploting when there is no more phosphosites

        # Seting subplots aprameters
        fig, ax = plt.subplots(sqrt_n_p, sqrt_n_p_X,
                               figsize=(18, 13))  # figsize=(7, 4) figsize=(18, 13) figsize=(29.7, 21)
        fig.tight_layout(w_pad=1.75, h_pad=3)
        plt.subplots_adjust(top=0.94)  # percentage of the figure that the plots are using

        # Go through the sub_df to plot all the phosphosites
        for i in range(sqrt_n_p):  # y
            for j in range(sqrt_n_p_X):  # X
                if k == number_phos:  # Stop plotting, all phosphorylation sites have been plotted
                    continue
                else:
                    # IN THIS CASE I AM ACCESSING THE ROW BY INDEXING,
                    # that's why the "id" is alreay at possition 0 and all the indexing is shifted compared to the datafram by -1
                    row = sub_df.iloc[k,
                          :]  # Go through the rows of the subdataset with the phsophosites for the protein
                    # Collect identification data of the phosphorylatio site
                    site = row[2]
                    name = row[1]
                    id = row[0]

                    # Collect data of the time points of the phosphosites
                    all_times = row[3:24]
                    EGF = row[3:10]
                    INS = row[10:17]
                    EGFnINS = row[17:24]

                    groups = [EGF, INS, EGFnINS]
                    colors = ['r', 'b', 'fuchsia']

                    # Collect data of the standard deviation of each timepoint
                    EGF_sd = row[26:33]
                    INS_sd = row[33:40]
                    EGFnINS_sd = row[40:]
                    groups_sd = [EGF_sd, INS_sd, EGFnINS_sd]
                    # Collect data about number of replicates in which the phosphosite was detected
                    n_rep = row[24]

                    # Start plotting
                    for c in range(3):
                        if n_rep == 1:
                            al = 0.3
                        else:
                            al = 1
                        ax[i, j].errorbar(x=['Full', '0', '1', '2', '5', '10', '90'], y=groups[c], yerr=groups_sd[c],
                                          marker='o', color=colors[c], alpha=al, capsize=4, elinewidth=1.3)

                        # Subplot parameters
                    ax[i, j].set_title(
                        f"{str(re.findall(r'^.*~', site))[2:-3]}_n_reps{n_rep}")  # , weight='bold'  INCLUDE THE POSIBILITY OF ADDING THE PHOSPHOSITE SEQUENCE
                    ax[i, j].set_xlabel("Time (min)")
                    ax[i, j].set_ylabel("Log2FC")
                    ax[i, j].grid()

                    # Using specific limits
                    if fit_y_lims == True:
                        ax[i, j].set_ylim(min(all_times) * 1.1 - 0.1, max(all_times) * 1.1 + 0.1)
                    else:  # Using general limits
                        ax[i, j].set_ylim(y_lim_min, y_lim_max)
                    ax[i, j].set_xlim(-1, 7)

                    # count the phosphorylation site as plotted
                    k = k + 1

        # General parameters of the plot
        fig.legend(labels=legend, loc="upper right", ncol=len(groups))
        fig.suptitle(f"{name}_{id}", weight='bold')

        # Saving the plot
        if save_pdf == True:
            plt.savefig(f"{to_save}/{protein}/{name}_{id}_0_All{y_fixed}{saving_info}.pdf")
            print(f"{name}_{id}_0_All{y_fixed}{saving_info}.pdf Plot saved as PDF")
        else:
            print(f"{protein} Plot not saved")
        if save_png == True:
            plt.savefig(f"{to_save}/{protein}/{name}_{id}_0_All{y_fixed}{saving_info}.png")
            print(f"{name}_{id}_0_All{y_fixed}{saving_info}.png Plot saved as PNG")
        if plot_close == True:
            plt.close()


def plot_protein_phosphosites_individually(df, proteins=list, legend=list, path=str, saving_path=str,
                                           saving_info="", save_pdf=False, save_png=False, plot_close=False,
                                           fit_y_lims=True):
    '''Create a PDF for each phosphorylation sites of a protein'''
    # Check the saving folder exists
    saving_folder = re.sub("Data.*", saving_path, path)
    if not os.path.exists(saving_folder):
        print("Creating saving folder")
        os.makedirs(saving_folder)

    for protein in proteins:

        # Create sub-dataframe with only the protein we are interested in. If the protein doesn't exist in the dataframe skip code
        if protein in df['prot_name'].to_list():
            sub_df = df.loc[df['prot_name'] == protein].copy()
            print(f"Ploting sites of protein {protein}")
        elif protein in df['protein_ID'].to_list():
            sub_df = df.loc[df['protein_ID'] == protein].copy()
            print(f"Ploting sites of protein {protein}")
        else:
            print(f"The protein {protein} is not present in the dataset")
            continue
        # Check if a folder for the desired protein exists. If no, create one
        to_save = re.sub("Data.*", saving_path, path)
        if protein in os.listdir(to_save):
            pass
        else:
            new_path = f"{to_save}/{protein}"
            os.makedirs(new_path)

        for row in sub_df.itertuples():
            # HERE THE ROW IS TAKEN OUT OF THE SUBDATAFRAME.
            # This is why position "0" is not usefull, is just the index in the dataframe
            site = row[3]
            name = row[2]
            id = row[1]

            all_times = row[4:25]
            EGF = row[4:11]
            INS = row[11:18]
            EGFnINS = row[18:25]
            groups = [EGF, INS, EGFnINS]

            # Collect data of the standard deviation of each timepoint
            EGF_sd = row[27:34]
            INS_sd = row[34:41]
            EGFnINS_sd = row[41:]
            groups_sd = [EGF_sd, INS_sd, EGFnINS_sd]

            # Collect data about number of replicates in which the phosphosite was detected
            n_rep = row[25]

            colors = ['r', 'b', 'fuchsia']

            fig, ax = plt.subplots(figsize=(7, 4))

            for c in range(3):
                if n_rep == 1:
                    al = 0.3
                else:
                    al = 1
                ax.errorbar(x=['Full', '0', '1', '2', '5', '10', '90'], y=groups[c], yerr=groups_sd[c], marker='o',
                            color=colors[c], label=legend[c], capsize=4, elinewidth=1.3, alpha=al)

            ax.set_title(f"{name}_{site}_n_rep{n_rep}")
            ax.set_xlabel("Time (min)")
            ax.set_ylabel("Log2FC")
            ax.set_xlim(-1, 7)
            if fit_y_lims == True:
                ax.set_ylim(min(all_times) * 1.1 - 0.1, max(all_times) * 1.1 + 0.1)
                y_lim = ""
            elif type(fit_y_lims) == list:
                ax.set_ylim(fit_y_lims[0], fit_y_lims[1])
                y_lim = f"_y_axis_fixed_{fit_y_lims[0]}_{fit_y_lims[1]}"

            ax.legend()
            ax.grid()

            if save_pdf == True:
                plt.savefig(f"{to_save}/{protein}/{name}_{site}{y_lim}{saving_info}.pdf")
                print(f"{name}_{site}{y_lim}{saving_info}.pdf Plot saved as PDF")
            else:
                print(f"{site} Plot not saved")
            if save_png == True:
                plt.savefig(f"{to_save}/{protein}/{name}_{site}{y_lim}{saving_info}.png")
                print(f"{name}_{site}{y_lim}{saving_info}.png Plot saved as PNG")
            if plot_close == True:
                plt.close()


def plot_dataset_phosphosites_together(df, proteins=list, legend=list, path=str, saving_path=str, dataset_folder=str,
                                       dataset_name=str,
                                       saving_info="", save_pdf=False, save_png=False, plot_close=False,
                                       fit_y_lims=True):
    '''Plot all the phosphorilation sites of a dataset'''
    # Check the saving folder exists
    saving_folder = re.sub("Data.*", saving_path, path)
    if not os.path.exists(saving_folder):
        print("Creating saving folder")
        os.makedirs(saving_folder)

    # Check if a folder for the desired protein exists. If no, create one
    to_save = re.sub("Data.*", saving_path, path)
    if dataset_name in os.listdir(to_save):
        pass
    else:
        new_path = f"{to_save}/{dataset_name}"
        os.makedirs(new_path)

    # Geting some basic information and parameters for the plots
    number_phos = len(df)
    sqrt_n_p = int(np.ceil(np.sqrt(number_phos)))
    if sqrt_n_p <= 2:
        empty_plots = 0
    else:
        empty_plots = (sqrt_n_p * sqrt_n_p) - number_phos
    # empty_plots = (sqrt_n_p * sqrt_n_p) - number_phos

    # Avoid getting rows with empty plots
    if empty_plots >= sqrt_n_p:
        sqrt_n_p_X = sqrt_n_p - 1
    else:
        sqrt_n_p_X = sqrt_n_p
        
    # number_phos = len(sub_df)
    # sqrt_n_p = int(np.ceil(np.sqrt(number_phos)))
    # if sqrt_n_p <= 2:
    #     empty_plots = 0
    # else:
    #     empty_plots = (sqrt_n_p * sqrt_n_p) - number_phos

    # Decide wether to fit the y axes or not
    if fit_y_lims == True:  # Fit "y" limits for each phosphosite
        y_fixed = "_y_axis_fixed"
    elif type(fit_y_lims) == list:
        y_lim_max = fit_y_lims[1]
        y_lim_min = fit_y_lims[0]
        y_fixed = f"_y_axis_fixed_{y_lim_min}_{y_lim_max}"
    else:  # Fit the same "y" limit for all phosphosites
        sub_values_df = df.iloc[:, 3:24] ######################################### 47?
        y_lim_max = (sub_values_df.max().max()) * 1.1
        y_lim_min = (sub_values_df.min().min()) * 1.1
        y_fixed = "_Y_fixed_general"

    k = 0  # Counter to stop ploting when there is no more phosphosites
    # Seting subplots parameters
    fig, ax = plt.subplots(sqrt_n_p, sqrt_n_p_X, figsize=(18, 13))  # figsize=(7, 4) figsize=(18, 13) figsize=(29.7, 21)
    fig.tight_layout(w_pad=1.75, h_pad=3)
    plt.subplots_adjust(top=0.94)  # percentage of the figure that the plots are using

    # Go through the sub_df to plot all the phosphosites
    for i in range(sqrt_n_p):  # y
        for j in range(sqrt_n_p_X):  # X
            if k == number_phos:  # Stop plotting, all phosphorylation sites have been plotted
                continue
            else:
                row = df.iloc[k, :]  # Go through the rows of the subdataset with the phsophosites for the protein
                # Collect identification data of the phosphorylatio site
                site = row[2]
                name = row[1]
                id = row[0]
                # Collect data of the time points of the phosphosites
                all_times = row[3:24]
                EGF = row[3:10]
                INS = row[10:17]
                EGFnINS = row[17:24]

                groups = [EGF, INS, EGFnINS]
                colors = ['r', 'b', 'fuchsia']

                # Collect data of the standard deviation of each timepoint

                EGF_sd = row[26:33]
                INS_sd = row[33:40]
                EGFnINS_sd = row[40:47]
                groups_sd = [EGF_sd, INS_sd, EGFnINS_sd]
                # print(groups, groups_sd)
                # Collect data about number of replicates in which the phosphosite was detected
                n_rep = row[24]

                # Start plotting
                for c in range(3):
                    if n_rep == 1:
                        al = 0.3
                    else:
                        al = 1
                    ax[i, j].errorbar(x=['Full', '0', '1', '2', '5', '10', '90'], y=groups[c], yerr=groups_sd[c],
                                      marker='o', color=colors[c], alpha=al, capsize=4, elinewidth=1.3)

                # Subplot parameters
                ax[i, j].set_title(
                    f"{str(re.findall(r'^.*~', site))[2:-3]}_n_reps{n_rep}")  # , weight='bold'  INCLUDE THE POSIBILITY OF ADDING THE PHOSPHOSITE SEQUENCE
                ax[i, j].set_xlabel("Time (min)")
                ax[i, j].set_ylabel("Log2FC")
                ax[i, j].grid()
                # Using specific limits
                if fit_y_lims == True:
                    ax[i, j].set_ylim(min(all_times) * 1.1 - 0.1, max(all_times) * 1.1 + 0.1)
                else:  # Using general limits
                    ax[i, j].set_ylim(y_lim_min, y_lim_max)
                ax[i, j].set_xlim(-1, 7)

                # count the phosphorylation site as plotted
                k = k + 1
    # General parameters of the plot
    fig.legend(labels=legend, loc="upper right", ncol=len(groups))
    fig.suptitle(f"{dataset_name}", weight='bold')

    # Saving the plot
    if save_pdf == True:
        plt.savefig(f"{to_save}/{dataset_folder}/{dataset_name}_All{y_fixed}{saving_info}.pdf")
        print(f"{dataset_name}_All{y_fixed}{saving_info}.pdf Plot saved as PDF")
    else:
        print(f"{dataset_name} Plot not saved")
    if save_png == True:
        plt.savefig(f"{to_save}/{dataset_folder}/{dataset_name}_All{y_fixed}{saving_info}.png")
        print(f"{dataset_name}_All{y_fixed}{saving_info}.png Plot saved as PNG")
    if plot_close == True:
        plt.close()


In [76]:
# Load the dataframe independently so it has not to be loaded each time we want to plot a set of proteins
path = "Experiment/1_hTERT_HME1/Data/Processed/Handmade_Log2_FC_from_FGZC.xlsx"
data_frame = pd.read_excel(path)
data_frame

,protein_ID,prot_name,site,EGF_full,EGF_starve,EGF1,EGF2,EGF5,EGF10,EGF90,...,INS5_std,INS10_std,INS90_std,EGFnINS_full_std,EGFnINS_starve_std,EGFnINS1_std,EGFnINS2_std,EGFnINS5_std,EGFnINS10_std,EGFnINS90_std
0,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_0~SGSGNFGGGR,1.341787,0,-0.045784,-0.057065,-0.181892,0.296514,0.152992,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_1_S154~SGsGNFGGGR,2.750617,0,-0.075632,0.051413,0.553841,1.071310,0.614675,...,0.132813,0.082451,0.158647,0.240242,0.073607,0.213038,0.017743,0.137722,0.158750,0.203929
2,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_0~NQGGYGGSSSSSSYGSGR,-0.475987,0,-0.055121,-0.172318,-0.206575,-0.181191,-0.458281,...,0.122319,0.050334,0.136001,0.110507,0.109621,0.049720,0.281925,0.100414,0.150921,0.137177
3,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S264~NQGGYGGSsSSSSYGSGR,-0.337808,0,0.099901,0.026794,-0.030970,-0.071683,-0.061284,...,0.229781,0.228683,0.101860,0.015341,0.096143,0.177090,0.027352,0.391594,0.279341,0.065133
4,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S266~NQGGYGGSSSsSSYGSGR,-0.726273,0,-0.103509,-0.389250,-0.566986,-0.520466,-0.750087,...,0.096032,0.092438,0.135065,0.190606,0.107901,0.087274,0.362086,0.085434,0.223696,0.084909
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34997,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_338_1_1_S338~SLsFEMQQDELIEK,0.030638,0,0.107473,-0.008990,0.060896,0.015639,0.004748,...,0.071151,0.118530,0.209883,0.140928,0.190284,0.162495,0.153906,0.077442,0.173945,0.194352
34998,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_0~SLSFEMQQDELIEKPMSPMQYAR,-0.325955,0,-0.094084,-0.149544,-0.103233,-0.104764,-0.135981,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34999,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_1_S338~SLsFEMQQDELIEKPMSPMQYAR,0.089820,0,0.002724,0.082102,0.025246,0.015418,0.111233,...,0.072786,0.092533,0.092859,0.181606,0.183772,0.186855,0.092704,0.159401,0.079835,0.147529
35000,Q9Y6Y8,SEC23IP,Q9Y6Y8_118_138_1_0~PLTALPFTTGSQDVSNAFSPSISK,-0.074021,0,-0.115939,-0.079316,0.204666,0.141208,0.312016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


["Q9H8S9", "Q7L9L4", "Q70IA8", "P46937", "Q16635", "O00750", "Q13043", "Q10325", "O54748", "O95835", "Q9NRM7"]

In [90]:
warnings.filterwarnings('ignore')

plot_protein_phosphosites_together(df= data_frame, proteins= ["EGFR"], #A0FGR8 
                                   legend = ['4.68 nM EGF', '10 nM INS', '4.68 nM EGF+ 10 nM INS'], 
                                   path= path, 
                                   saving_path = "Results/Interesting_proteins", 
                                   saving_info = "_Log2_FC_handmade_from_FGCZ", save_pdf = False,
                                   save_png = False, plot_close = True, fit_y_lims = False) 
# 
# plot_protein_phosphosites_individually(df= data_frame, proteins= ["MAPK1"], 
#                                        legend = ['4.68 nM EGF', '10 nM INS', '4.68 nM EGF+ 10 nM INS'], 
#                                        path= path,
#                                        saving_path = "Results/Interesting_proteins",
#                                        saving_info = "_Log2_FC_handmade_from_FGCZ", save_pdf = False,
#                                        save_png = False, plot_close = False, fit_y_lims = True)
# 
# plot_dataset_phosphosites_together(df=data_frame , legend = ['4.68 nM EGF', '10 nM INS', '4.68 nM EGF+ 10 nM INS'], 
#                                    path= "Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC_nrep_SD.xlsx", saving_path = "Results/Sub_datasets",
#                                    dataset_folder = "Testing", dataset_name = "Testing", 
#                                    saving_info = "", save_pdf = False, save_png = False, plot_close = True, fit_y_lims = False)

Ploting sites of protein EGFR
EGFR Plot not saved


## Plotting external data. 
### P146 A Phosphoproteomics Data Resource for Systems-level Modeling of Kinase Signaling Networks

data

In [83]:
# Load the dataframe independently so it has not to be loaded each time we want to plot a set of proteins
path = "External_Data/phospho_proteomics_data_P146/Data/timecourse_FC_data.xlsx"
data_frame_p146 = pd.read_excel(path)
data_frame_p146

,uniprot_ID,prot_name,residue,control_FC,EGF2_FC,EGF4_FC,EGF8_FC,EGF12_FC,control_std,EGF2_std,EGF4_std,EGF8_std,EGF12_std
0,A0A024RBG1,NUD4B_HUMAN,S149s,0.0,-0.151147,-0.177721,0.488392,0.235037,0.097040,0.070082,0.328498,0.004493,0.181938
1,A0A075B759,PAL4E_HUMAN,S51s,0.0,0.162495,-0.338475,-0.116367,-0.144441,0.232915,0.145111,0.164571,0.051806,0.172279
2,A0A075B759,PAL4E_HUMAN,T32t,0.0,0.365470,0.013353,-0.079198,-0.004393,0.266326,0.072477,0.225580,0.005211,0.014127
3,A0A0B4J1U4,TRGV5_HUMAN,S82sY87yT90t,0.0,0.047770,0.144271,0.453530,0.574164,0.102742,0.142777,0.304096,0.165791,0.070075
4,A0A0B4J2F2,SIK1B_HUMAN,S534s,0.0,-0.024795,0.287202,0.500912,0.156209,0.198640,0.213446,0.183640,0.213231,0.013910
...,...,...,...,...,...,...,...,...,...,...,...,...,...
45037,Q9Y6Y0,NS1BP_HUMAN,S338s,0.0,-0.172555,0.122489,0.190293,0.323714,0.140480,0.350749,0.010127,0.226493,0.043201
45038,Q9Y6Y0,NS1BP_HUMAN,S326s,0.0,0.487067,0.693506,0.737479,1.130546,0.028874,0.138680,0.009456,0.165527,0.268179
45039,Q9Y6Y8,S23IP_HUMAN,S926s,0.0,0.050537,0.147416,0.111000,0.263787,0.015108,0.240039,0.090636,0.211142,0.038269
45040,Q9Y6Y8,S23IP_HUMAN,S926s,0.0,-0.066078,0.332327,0.378860,0.536770,0.349860,0.251235,0.009028,0.185927,0.135118


In [84]:
def plot_protein_phosphosites_together_p146(df, proteins=list, legend=list, path=str, saving_path=str,
                                            saving_info="", save_pdf=False, save_png=False, plot_close=False,
                                            fit_y_lims=False):
    '''Plot to PDF ALL phosphosites of a protein together in one single plot. "y" limits are ajusted to the limits of each phosphosites'''
    # Check the saving folder exists
    saving_folder = re.sub("/Data/.*", f"/{saving_path}", path)
    if not os.path.exists(saving_folder):
        print("Creating saving folder")
        os.makedirs(saving_folder)

    for protein in proteins:

        # Create sub-dataframe with only the protein we are interested in. If the protein doesn't exist in the dataframe skip code
        if protein in df['prot_name'].to_list():
            sub_df = df.loc[df['prot_name'] == protein].copy()
            print(f"Ploting sites of protein {protein}")
        elif protein in df['uniprot_ID'].to_list():
            sub_df = df.loc[df['uniprot_ID'] == protein].copy()
            print(f"Ploting sites of protein {protein}")
        else:
            print(f"The protein {protein} is not present in the dataset")
            continue
        # Check if a folder for the desired protein exists. If no, create one
        if protein in os.listdir(saving_folder):
            pass
        else:
            new_path = f"{saving_folder}/{protein}"
            print(f"Createating saving folder for {protein}")
            os.makedirs(new_path)

        # sub_df.sort_values(by=['site'], inplace=True)

        # Geting some basic information and parameters for the plots
        number_phos = len(sub_df)
        sqrt_n_p = int(np.ceil(np.sqrt(number_phos)))
        if sqrt_n_p <= 2:
            empty_plots = 0
        else:
            empty_plots = (sqrt_n_p * sqrt_n_p) - number_phos
        # print(number_phos, sqrt_n_p, empty_plots)

        # If the protein only has one phosphosite the code cannot make a plot with one subplot, so I'll have to make it individually
        # single_phsite_proteins = []
        # if number_phos == 1:
        #     single_phsite_proteins.append(protein)
        #     print(f"Protein {protein} has only one phosphosite. Plot it individually")
        #     continue

        # Avoid getting rows with empty plots
        if empty_plots >= sqrt_n_p:
            sqrt_n_p_X = sqrt_n_p - 1
        else:
            sqrt_n_p_X = sqrt_n_p

        # Decide wether to fit the y axes or not
        if fit_y_lims == True:  # Fit "y" limits for each phosphosite
            y_fixed = "_y_axis_fixed"
        elif type(fit_y_lims) == list:
            y_lim_min = fit_y_lims[0]
            y_lim_max = fit_y_lims[1]
            y_fixed = f"_y_axis_fixed_{y_lim_min}_{y_lim_max}"
        else:  # Fit the same "y" limit for all phosphosites
            sub_values_df = sub_df.iloc[:, 4:9]
            y_lim_max = (sub_values_df.max().max()) * 1.1
            y_lim_min = (sub_values_df.min().min()) * 1.1
            if y_lim_min > 0:
                y_lim_min = -1 * y_lim_min
            y_fixed = "_y_axis_general"

        k = 0  # Counter to stop ploting when there is no more phosphosites

        # Seting subplots aprameters
        fig, ax = plt.subplots(sqrt_n_p, sqrt_n_p_X,figsize=(18, 13))  # figsize=(7, 4) figsize=(18, 13) figsize=(29.7, 21)
        fig.tight_layout(w_pad=1.75, h_pad=3)
        plt.subplots_adjust(top=0.94)  # percentage of the figure that the plots are using

        # Go through the sub_df to plot all the phosphosites
        for i in range(sqrt_n_p):  # y
            for j in range(sqrt_n_p_X):  # X
                if k == number_phos:  # Stop plotting, all phosphorylation sites have been plotted
                    continue
                else:
                    # IN THIS CASE I AM ACCESSING THE ROW BY INDEXING,
                    # that's why the "id" is alreay at possition 0 and all the indexing is shifted compared to the datafram by -1
                    row = sub_df.iloc[k,:]  # Go through the rows of the subdataset with the phsophosites for the protein
                    # Collect identification data of the phosphorylatio site
                    site = row[2] #2
                    name = row[1] #1
                    id = row[0] #0

                    # Collect data of the time points of the phosphosites
                    EGF = row[3:8]

                    # Collect data of the standard deviation of each timepoint
                    EGF_sd = row[8:]

                    ax[i, j].errorbar(x=['Controll', '2', '4', '8', '12'], y=EGF, yerr=EGF_sd,
                                      marker='o', color='r', capsize=4, elinewidth=1.3)

                        # Subplot parameters
                    ax[i, j].set_title(f"{site}")  # , weight='bold'  INCLUDE THE POSIBILITY OF ADDING THE PHOSPHOSITE SEQUENCE
                    ax[i, j].set_xlabel("Time (min)")
                    ax[i, j].set_ylabel("Log2FC")
                    ax[i, j].grid()

                    # Using specific limits
                    if fit_y_lims == True:
                        min_y = min(EGF)* 1.1 
                        max_y = max(EGF)* 1.1
                        if min_y > 0:
                            min_y = -1 * min_y
                        ax[i, j].set_ylim(min_y, max_y)
                    else:  # Using general limits
                        ax[i, j].set_ylim(y_lim_min, y_lim_max)
                    ax[i, j].set_xlim(-1, 5)

                    # count the phosphorylation site as plotted
                    k = k + 1

        # General parameters of the plot
        fig.legend(labels=legend, loc="upper right") # ncol=len(groups
        fig.suptitle(f"{name}_{id}", weight='bold')

        # Saving the plot
        if save_pdf == True:
            plt.savefig(f"{saving_folder}/{protein}/{name}_{id}_0_All{y_fixed}{saving_info}.pdf")
            print(f"{name}_{id}_0_All{y_fixed}{saving_info}.pdf Plot saved as PDF")
        else:
            print(f"{protein} Plot not saved")
        if save_png == True:
            plt.savefig(f"{saving_folder}/{protein}/{name}_{id}_0_All{y_fixed}{saving_info}.png")
            print(f"{saving_folder}/{protein}/{name}_{id}_0_All{y_fixed}{saving_info}.png Plot saved as PNG")
        if plot_close == True:
            plt.close()

In [86]:
plot_protein_phosphosites_together_p146(df= data_frame_p146, proteins= ["P00533"], #A0FGR8 P28482
                                        legend = ['X nM EGF'], 
                                        path= path, 
                                        saving_path = "Results/Interesting_proteins", 
                                        saving_info = "_Log2_FC", save_pdf = False,
                                        save_png = False, plot_close = True, fit_y_lims = False) 
# 

Ploting sites of protein P00533
P00533 Plot not saved
